In [ ]:
SHARD, SHARDS = 0, 1  # run once per shard index for parallel sessions
VERSION = "17.1"
DRIVE = "/content/drive/MyDrive/chessdb"  # checkpoint folder on Drive
DB = f"/content/chessdb-s{SHARD}.sqlite"

from google.colab import drive

drive.mount("/content/drive")

!git -C /content/chessdb pull || git clone https://github.com/abstratovcm/chessdb.git /content/chessdb
%cd /content/chessdb
!pip install -e . -q
!chessdb fetch-engine --version $VERSION
!PYTHONUNBUFFERED=1 chessdb run --db $DB --version $VERSION --drive-dir $DRIVE --max-hours 8 --seed 1 --games-target 2000 --top-rung 4000000 --checkpoint-minutes 5 --shards $SHARDS --shard-index $SHARD

In [ ]:
!nproc
!chessdb init --db $DB
!ls -la $DB
!ls -la "$DRIVE/"

In [ ]:
!chessdb metrics --db $DB

import sys
from pathlib import Path

# the editable install is only visible after a runtime restart
sys.path.insert(0, "/content/chessdb/src")
from chessdb.db.checkpoint import upload_checkpoint

dest = Path(DRIVE) / f"s{SHARD}of{SHARDS}" if SHARDS > 1 else Path(DRIVE)
upload_checkpoint(Path(DB), dest)